# Video HLS Generator — Kaggle Edition

Converts videos to adaptive HLS streams (multi-resolution).

**How to run unattended (tab-close safe):**
1. Add your videos as a Kaggle Dataset (or set a URL below)
2. Set `DATASET_PATH` or `SINGLE_VIDEO_URL` in the Config cell
3. Click **Save & Run All (Commit)** — Kaggle runs the full notebook in the background
4. Output files appear in the **Output** tab when done

> **Tip:** Enable the T4 GPU under *Session options → Accelerator → GPU T4 x2*

In [ ]:
# ── Detect GPU & install extras ───────────────────────────────────────────────
import subprocess

# ffmpeg is pre-installed on Kaggle; install yt-dlp + gdown for URL downloads
subprocess.run(["pip", "install", "-q", "yt-dlp", "gdown"], check=False)

caps = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True)
USE_GPU = "h264_nvenc" in caps.stdout

if USE_GPU:
    print("✓ GPU detected — CUDA decode + NVENC encode (fast)")
else:
    print("⚠ No GPU — using libx264 (CPU). Enable T4 in Session options.")

In [ ]:
# ── Configuration — edit these ────────────────────────────────────────────────

# Option A: Kaggle dataset containing your videos
# Upload your videos at kaggle.com/datasets/new, then add the dataset to this
# notebook via the + Add Data button. The path will be /kaggle/input/<dataset-name>/
DATASET_PATH = "/kaggle/input/"   # set to exact dataset folder, e.g. "/kaggle/input/my-videos/"

# Option B: Single direct video URL (YouTube, direct .mp4 link, etc.)
# Set DATASET_PATH = "" and fill this instead
SINGLE_VIDEO_URL = ""

# Option C: Google Drive direct file URL (single file, must be publicly shared)
# Set DATASET_PATH = "" and SINGLE_VIDEO_URL = "" and fill this instead
GDRIVE_FILE_ID = ""   # e.g. "1A2B3C4D5E6F..." from the share link

SELECTED_RESOLUTIONS = ["1080p", "720p", "480p", "360p"]  # remove any you don't need

CPU_THREADS = 4   # only used when GPU is not available

In [ ]:
# ── Discover / download input videos ─────────────────────────────────────────
import os, re, shutil
from pathlib import Path

VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".flv", ".m4v"}
INPUT_DIR  = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working/hls_output"
DOWNLOAD_DIR = "/kaggle/working/input_videos"
os.makedirs(OUTPUT_DIR, exist_ok=True)

video_files = []

if DATASET_PATH and DATASET_PATH != "/kaggle/input/":
    # Scan the specified dataset folder
    for p in sorted(Path(DATASET_PATH).rglob("*")):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS:
            video_files.append(str(p))
    if not video_files:
        raise RuntimeError(f"No video files found in {DATASET_PATH}")

elif SINGLE_VIDEO_URL:
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    print(f"Downloading: {SINGLE_VIDEO_URL}")
    result = subprocess.run(
        ["yt-dlp", "--no-playlist", "-o", f"{DOWNLOAD_DIR}/%(title)s.%(ext)s", SINGLE_VIDEO_URL],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        filename = SINGLE_VIDEO_URL.split("/")[-1].split("?")[0] or "video.mp4"
        subprocess.run(["wget", "-q", "-O", f"{DOWNLOAD_DIR}/{filename}", SINGLE_VIDEO_URL])
    video_files = sorted([
        str(p) for p in Path(DOWNLOAD_DIR).iterdir()
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    ])

elif GDRIVE_FILE_ID:
    import gdown
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    out_path = f"{DOWNLOAD_DIR}/gdrive_video.mp4"
    print(f"Downloading from Google Drive: {GDRIVE_FILE_ID}")
    gdown.download(id=GDRIVE_FILE_ID, output=out_path, quiet=False)
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        video_files = [out_path]
    else:
        raise RuntimeError("Drive download failed or file is 0 bytes.")

else:
    raise ValueError("Set DATASET_PATH, SINGLE_VIDEO_URL, or GDRIVE_FILE_ID in the config cell.")

print(f"\n✓ {len(video_files)} video(s) ready to process:")
for v in video_files:
    print(f"  • {Path(v).name}")

In [ ]:
# ── HLS Generation ────────────────────────────────────────────────────────────

RESOLUTIONS = {
    "1080p": {"width": 1920, "height": 1080, "bitrate": "5000k", "audio": "192k"},
    "720p":  {"width": 1280, "height": 720,  "bitrate": "2800k", "audio": "128k"},
    "480p":  {"width": 854,  "height": 480,  "bitrate": "1400k", "audio": "128k"},
    "360p":  {"width": 640,  "height": 360,  "bitrate": "800k",  "audio": "96k"},
}

MASTER_BANDWIDTHS = {
    "1080p": 5000000, "720p": 2800000, "480p": 1400000, "360p": 800000,
}

def _double_bitrate(bitrate: str) -> str:
    return str(int(bitrate.rstrip("k")) * 2) + "k"


def encode_video(video_path, selected_res):
    stem = Path(video_path).stem
    job_dir = os.path.join(OUTPUT_DIR, stem)
    os.makedirs(job_dir, exist_ok=True)

    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", video_path],
        capture_output=True, text=True
    )
    try:
        duration = float(probe.stdout.strip())
        print(f"  Duration: {duration:.1f}s")
    except Exception:
        duration = 0.0

    thumb_path = os.path.join(job_dir, "thumbnail.jpg")
    subprocess.run(
        ["ffmpeg", "-y", "-ss", "1", "-i", video_path,
         "-vframes", "1", "-q:v", "2", thumb_path],
        capture_output=True
    )

    for res in selected_res:
        os.makedirs(os.path.join(job_dir, res), exist_ok=True)

    n = len(selected_res)

    # Full GPU pipeline: hwaccel_output_format cuda keeps frames in VRAM so
    # scale_npp runs entirely on GPU — no CPU round-trip between decode and encode.
    scale_filter = "scale_npp" if USE_GPU else "scale"
    split_labels = "".join(f"[v{i}]" for i in range(n))
    filter_parts = [f"[0:v]split={n}{split_labels}"]
    for i, res in enumerate(selected_res):
        cfg = RESOLUTIONS[res]
        filter_parts.append(f"[v{i}]{scale_filter}={cfg['width']}:{cfg['height']}[s{i}]")
    filter_complex = ";".join(filter_parts)

    hwaccel_flags = ["-hwaccel", "cuda", "-hwaccel_output_format", "cuda"] if USE_GPU else []

    base_cmd = ["ffmpeg", "-y", *hwaccel_flags, "-i", video_path,
                "-filter_complex", filter_complex]
    if not USE_GPU:
        base_cmd += ["-threads", str(CPU_THREADS)]

    per_output = []
    for i, res in enumerate(selected_res):
        cfg = RESOLUTIONS[res]
        segment_pat = os.path.join(job_dir, res, "%03d.ts")
        playlist    = os.path.join(job_dir, res, "playlist.m3u8")

        if USE_GPU:
            video_flags = ["-c:v", "h264_nvenc", "-preset", "p4",
                           "-rc", "vbr", "-b:v", cfg["bitrate"], "-maxrate", cfg["bitrate"],
                           "-b_ref_mode", "disabled",
                           "-profile:v", "main"]
        else:
            video_flags = ["-c:v", "libx264", "-crf", "23", "-preset", "medium",
                           "-profile:v", "main",
                           "-maxrate", cfg["bitrate"], "-bufsize", _double_bitrate(cfg["bitrate"])]

        per_output += [
            "-map", f"[s{i}]", "-map", "0:a?",
            *video_flags,
            "-c:a", "aac", "-b:a", cfg["audio"],
            "-hls_time", "10", "-hls_playlist_type", "vod",
            "-hls_segment_filename", segment_pat,
            playlist,
        ]

    cmd = base_cmd + per_output

    res_label = "+".join(selected_res)
    encoder_label = "NVENC (full GPU)" if USE_GPU else f"libx264 ({CPU_THREADS}t)"
    print(f"  Encoding [{res_label}] via {encoder_label}...", end="", flush=True)

    process = subprocess.Popen(cmd, stderr=subprocess.PIPE, stdout=subprocess.DEVNULL,
                               text=True, bufsize=1)

    stderr_lines = []
    last_pct = -1
    for line in process.stderr:
        stderr_lines.append(line)
        m = re.search(r"time=(\d+):(\d+):([\d.]+)", line)
        if m and duration:
            elapsed = int(m.group(1))*3600 + int(m.group(2))*60 + float(m.group(3))
            pct = min(int(elapsed / duration * 100), 100)
            if pct != last_pct and pct % 10 == 0:
                print(f" {pct}%", end="", flush=True)
                last_pct = pct

    process.wait()
    if process.returncode != 0:
        print(f"\n[FFmpeg failed — exit {process.returncode}]")
        print("".join(stderr_lines[-40:]))
        raise RuntimeError(f"FFmpeg encoding failed (exit {process.returncode})")
    print(" ✓")

    master_path = os.path.join(job_dir, "master.m3u8")
    with open(master_path, "w") as f:
        f.write("#EXTM3U\n#EXT-X-VERSION:3\n")
        for res in selected_res:
            cfg = RESOLUTIONS[res]
            f.write(f"#EXT-X-STREAM-INF:BANDWIDTH={MASTER_BANDWIDTHS[res]},"
                    f"RESOLUTION={cfg['width']}x{cfg['height']}\n")
            f.write(f"{res}/playlist.m3u8\n")

    print(f"  ✓ master.m3u8 → {master_path}")
    return job_dir


print(f"Processing {len(video_files)} file(s):\n")
completed = []

for idx, video_path in enumerate(video_files):
    stem = Path(video_path).stem
    print(f"[{idx+1}/{len(video_files)}] {stem}")
    job_dir = encode_video(video_path, SELECTED_RESOLUTIONS)
    completed.append((stem, job_dir))
    print(f"  ✓ Done → {job_dir}\n")

print(f"=== All {len(completed)} video(s) processed ===")

In [ ]:
# ── Zip output for easy download ──────────────────────────────────────────────
# Files in /kaggle/working/ are always available in the Output tab.
# This cell zips everything into one archive for convenience.

if len(completed) == 1:
    stem, job_dir = completed[0]
    zip_path = shutil.make_archive(f"/kaggle/working/{stem}_hls", "zip", OUTPUT_DIR, stem)
else:
    zip_path = shutil.make_archive("/kaggle/working/hls_output_all", "zip", "/kaggle/working", "hls_output")

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"✓ Zipped → {zip_path}  ({size_mb:.1f} MB)")
print("\nDownload from the Output tab on the right →")